# Experiments runner

Batch runner for the numerical experiments. The scripts are divided into four main parts according to the solver being used: 
- MUMPS
- SUPERLU_DIST
- PASTIX
- MKL PARDISO

Solver-specific options are set for all of these scripts.

Set `-own_reordering true` to apply reordering via the custom `reorder()` routine before solving.  
Set `-own_reordering false` (default) to let the solver handle reordering internally.

---

## Configuration

In [1]:
import os
import subprocess

In [ ]:
# data paths
MAT_INPUT_PATH = "../matrices/bin/"
VEC_INPUT_PATH = "../matrices/bin/vec/"
MAT_PERM_PATH = "../matrices/bin/permuted/"

# PETSc builds
PETSC_DIR          = "../petsc"
PETSC_ARCH_DEFAULT = "arch-linux-c-opt"
PETSC_ARCH_PARDISO = "arch-mklp"

# executables
# - EXE_DEFAULT uses main PETSc build (arch-linux-c-opt)
# - EXE_PARDISO uses the dedicated MKL/PARDISO PETSc build (arch-mklp)

EXE_DEFAULT = f"{PETSC_DIR}/{PETSC_ARCH_DEFAULT}/bin/mpirun -n {{nproc}} ../src/experiment"
EXE_PARDISO = (
    "LD_PRELOAD=/opt/intel/oneapi/mkl/2025.3/lib/libmkl_blacs_intelmpi_lp64.so "
    "/opt/intel/oneapi/mpi/2021.16/bin/mpiexec -n {nproc} ../src/experiment_pardiso"
)

## Helper function

In [ ]:
def run_experiments(
        exe, 
        solver, 
        nproc, 
        solver_options, 
        log_dir, 
        ordering_tag, 
        pc_type="lu",
        own_reordering=False, 
        ordering_type=None, 
        rhs_file=None):
    """
    Run the experiment executable over all .bin matrices in MAT_INPUT_PATH.

    Parameters
    ----------
    exe             : str   Executable template with {nproc} placeholder.
    solver          : str   PETSc mat_solver_type (e.g. 'mumps').
    nproc           : int   Number of MPI processes.
    solver_options  : str   Solver-specific PETSc flags as a single string.
    log_dir         : str   Directory where log files are saved.
    ordering_tag    : str   Short label used in the log filename.
    pc_type         : str   Preconditioner type (LU by default).
    own_reordering  : bool  If True, use custom reorder() before solving.
    ordering_type   : str   PETSc ordering type (only used if own_reordering=True).
    rhs_file        : str   Optional path to a .bin RHS vector.
    """
    os.makedirs(log_dir, exist_ok=True)

    mat_files = sorted(f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin"))

    for mat_file in mat_files:
        mat_name = os.path.splitext(mat_file)[0]
        mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
        log_file = f"{ordering_tag}_{mat_name}_{solver}_{nproc}procs.log"
        log_path = os.path.join(log_dir, log_file)

        cmd_parts = [
            exe.format(nproc=nproc),
            f"-mat_file {mat_path}",
            f"-pc_type {pc_type}",
            f"-mat_solver_type {solver}",
            "-get_solution_norm true",
        ]

        if rhs_file:
            cmd_parts.append(f"-rhs_file {rhs_file}")

        if own_reordering and ordering_type:
            cmd_parts += [
                "-own_reordering true",
                f"-mat_ordering_type {ordering_type}",
            ]

        if solver_options:
            cmd_parts.append(solver_options)

        cmd = " ".join(cmd_parts)

        print(f"Running {mat_file} -> {log_file}")
        with open(log_path, "w") as f:
            subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

    print("======= COMPLETE =======")

## MUMPS runs

In [ ]:
MUMPS_OPTIONS = (
    "-mat_mumps_icntl_2 3 "    # diagnostics to stderr
    "-mat_mumps_icntl_4 4 "    # maximum print level
    "-mat_mumps_icntl_7 6 "    # sequential ordering: QAMD
    "-mat_mumps_icntl_28 2 "   # parallel analysis
    "-mat_mumps_icntl_29 1 "   # parallel ordering: PT-Scotch
    "-mat_mumps_icntl_22 1"    # out-of-core
)

run_experiments(
    exe           = EXE_DEFAULT,
    solver        = "mumps",
    nproc         = 4,
    solver_options= MUMPS_OPTIONS,
    log_dir       = "./mumps",
    ordering_tag  = "qamd",
)

Running usroads.bin -> qamd_usroads_mumps_4procs.log
======= COMPLETE =======


## SUPERLU_DIST runs

In [7]:
SUPERLU_OPTIONS = (
    "-mat_superlu_dist_rowperm NOROWPERM "  # NOROWPERM | LargeDiag_MC64 | LargeDiag_AWPM | MY_PERMR
    "-mat_superlu_dist_colperm NATURAL "    # NATURAL | MMD_AT_PLUS_A | MMD_ATA | METIS_AT_PLUS_A | PARMETIS
    "-mat_superlu_dist_printstat"           # print factorization info
)

run_experiments(
    exe           = EXE_DEFAULT,
    solver        = "superlu_dist",
    nproc         = 8,
    solver_options= SUPERLU_OPTIONS,
    log_dir       = "./superlu_dist",
    ordering_tag  = "natural",
)

Running usroads.bin -> natural_usroads_superlu_dist_8procs.log
======= COMPLETE =======


## PaStiX runs

In [ ]:
PASTIX_OPTIONS = (
    "-mat_pastix_verbose 2 "        # maximum verbosity
    "-mat_pastix_factorization 2 "  # LU
    "-mat_pastix_ordering 1 "       # Metis
    "-mat_pastix_thread_nbr 1"      # 1 thread per MPI process
)

run_experiments(
    exe           = EXE_DEFAULT,
    solver        = "pastix",
    nproc         = 8,
    solver_options= PASTIX_OPTIONS,
    log_dir       = "./pastix",
    ordering_tag  = "metis",
)

Running usroads.bin -> metis_usroads_pastix_8procs.log
======= COMPLETE =======


## MKL PARDISO

Uses a separate PETSc build (`arch-mklp`) and Intel MPI. Set `use_external_ordering = True`
to apply a custom reordering before passing the matrix to PARDISO.

PETSc solver type:
- `mkl_cpardiso` — MPI (Cluster PARDISO)
- `mkl_pardiso`  — sequential

In [ ]:
use_external_ordering = False
ordering_type         = "nd"          # used only if use_external_ordering=True
mat_solver            = "mkl_cpardiso"

PARDISO_OPTIONS = ""

ordering_tag = f"external_{ordering_type}" if use_external_ordering else "internal"

run_experiments(
    exe             = EXE_PARDISO,
    solver          = mat_solver,
    nproc           = 8,
    solver_options  = PARDISO_OPTIONS,
    log_dir         = "./pardiso",
    ordering_tag    = ordering_tag,
    own_reordering  = use_external_ordering,
    ordering_type   = ordering_type if use_external_ordering else None,
)

Running usroads.bin -> internal_usroads_mkl_cpardiso_8procs.log
======= COMPLETE =======
